# 02 - Tratamento E Validacao Dos Dados

Etapa de preparacao da base analitica do projeto Case Sorveteria Analytics.

Objetivos:

- corrigir inconsistencias;
- tratar valores nulos;
- padronizar colunas;
- converter tipos corretamente;
- criar colunas auxiliares para analise;
- preparar a base para Power BI;
- garantir qualidade, rastreabilidade e governanca dos dados.

Regra de governanca: nenhum arquivo em `data/raw` e alterado.

In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import data_cleaning

PROJECT_ROOT

WindowsPath('C:/Users/User/OneDrive/Documentos/David/case-sorveteria-analytics')

## Execucao Do Pipeline

O pipeline le a base bruta, cria uma camada intermediaria auditavel, separa registros excluidos e grava a base processada.

In [2]:
resultado = data_cleaning.run_pipeline(write_outputs=True)

raw = resultado.raw
interim = resultado.interim
processed = resultado.processed
rejected = resultado.rejected
quality_report = resultado.quality_report

raw.shape, interim.shape, processed.shape, rejected.shape

((50000, 12), (50000, 45), (48491, 31), (1509, 45))

## Arquivos Gerados

In [3]:
arquivos_gerados = pd.DataFrame([
    {"camada": "interim", "arquivo": str(data_cleaning.INTERIM_OUTPUT_PATH.relative_to(PROJECT_ROOT))},
    {"camada": "interim", "arquivo": str(data_cleaning.REJECTED_OUTPUT_PATH.relative_to(PROJECT_ROOT))},
    {"camada": "interim", "arquivo": str(data_cleaning.QUALITY_REPORT_PATH.relative_to(PROJECT_ROOT))},
    {"camada": "processed", "arquivo": str(data_cleaning.PROCESSED_OUTPUT_PATH.relative_to(PROJECT_ROOT))},
])
arquivos_gerados

,camada,arquivo
0,interim,data\interim\vendas_sorvetes_interim.csv
1,interim,data\interim\vendas_sorvetes_registros_excluidos.csv
2,interim,data\interim\relatorio_qualidade_tratamento.csv
3,processed,data\processed\vendas_sorvetes_tratado.csv


## Relatorio De Qualidade

Resumo do impacto das regras de tratamento e validacao.

In [4]:
quality_report

,metrica,valor,percentual_base
0,linhas_base_bruta,50000,100.00
1,linhas_base_processada,48491,96.98
2,linhas_removidas_base_processada,1509,3.02
3,celulas_texto_padronizadas,22822,NaN
4,sabor_nulo_preenchido,500,1.00
5,cidade_nula_preenchida,500,1.00
6,valor_total_nulo,500,1.00
7,quantidade_invalida,1016,2.03
8,valor_total_invalido,1509,3.02
9,data_invalida,0,0.00


## Linhas Removidas

In [5]:
linhas_removidas = int((~interim["registro_valido_powerbi"]).sum())
percentual_removido = linhas_removidas / len(interim) * 100

pd.DataFrame([
    {"metrica": "linhas_originais", "valor": len(raw)},
    {"metrica": "linhas_processadas", "valor": len(processed)},
    {"metrica": "linhas_removidas", "valor": linhas_removidas},
    {"metrica": "percentual_removido", "valor": round(percentual_removido, 2)},
])

,metrica,valor
0,linhas_originais,"50,000.00"
1,linhas_processadas,"48,491.00"
2,linhas_removidas,"1,509.00"
3,percentual_removido,3.02


## Motivos De Exclusao

Um registro pode ter mais de um motivo de exclusao. A base de auditoria preserva todos os casos removidos da camada processada.

In [6]:
motivos = (
    rejected["motivo_exclusao"]
    .str.get_dummies(sep=";")
    .sum()
    .sort_values(ascending=False)
    .rename_axis("motivo")
    .reset_index(name="registros")
)
motivos

,motivo,registros
0,quantidade_nao_positiva_ou_nula,1016
1,valor_total_nao_positivo,1009
2,valor_total_nulo,500


## Qualidade Final Da Base Processada

In [7]:
qualidade_final = pd.DataFrame([
    {"checagem": "nulos totais", "resultado": int(processed.isna().sum().sum())},
    {"checagem": "ids transacao duplicados", "resultado": int(processed["id_transacao"].duplicated().sum())},
    {"checagem": "quantidade_vendida <= 0", "resultado": int((processed["quantidade_vendida"] <= 0).sum())},
    {"checagem": "receita_transacao <= 0", "resultado": int((processed["receita_transacao"] <= 0).sum())},
    {"checagem": "datas invalidas", "resultado": int(pd.to_datetime(processed["data_venda"], errors="coerce").isna().sum())},
    {"checagem": "linhas validas para Power BI", "resultado": int(processed["flag_registro_valido_powerbi"].sum())},
])
qualidade_final

,checagem,resultado
0,nulos totais,0
1,ids transacao duplicados,0
2,quantidade_vendida <= 0,0
3,receita_transacao <= 0,0
4,datas invalidas,0
5,linhas validas para Power BI,48491


## Colunas Criadas Para Analise Executiva E Power BI

In [8]:
colunas_auxiliares = [
    "ano", "mes", "nome_mes", "ano_mes", "trimestre", "dia_semana", "dia_mes",
    "hora_venda", "hora", "faixa_horaria", "valor_transacao", "valor_unitario_medio",
    "valor_unitario_estimado", "status_promocao", "cliente_recorrente",
    "quantidade_transacoes_cliente", "flag_sabor_nao_informado", "flag_cidade_nao_informada",
    "flag_outlier_valor_total", "flag_registro_valido_powerbi",
]

pd.DataFrame({"coluna_auxiliar": colunas_auxiliares})

,coluna_auxiliar
0,ano
1,mes
2,nome_mes
3,ano_mes
4,trimestre
5,dia_semana
6,dia_mes
7,hora_venda
8,hora
9,faixa_horaria


## Amostra Da Base Processada

In [9]:
processed.head()

,id_transacao,data_venda,ano,mes,nome_mes,ano_mes,trimestre,dia_semana,dia_mes,hora_venda,hora,faixa_horaria,tipo_sorvete,sabor,quantidade_vendida,receita_transacao,valor_transacao,valor_unitario_medio,valor_unitario_estimado,cidade,estado,canal_venda,promocao,status_promocao,id_cliente,cliente_recorrente,quantidade_transacoes_cliente,flag_sabor_nao_informado,flag_cidade_nao_informada,flag_outlier_valor_total,flag_registro_valido_powerbi
0,1,2025-04-30,2025,4,Abril,2025-04,T2,Quarta,30,11:00,11,Manha,Milkshake,Açaí,3,16.22,16.22,5.41,5.41,Campinas,AP,App,True,Com Promocao,CLI9795,True,3,False,False,False,True
1,2,2025-07-12,2025,7,Julho,2025-07,T3,Sabado,12,15:45,15,Tarde,Milkshake,Menta,4,18.57,18.57,4.64,4.64,Aragão,TO,App,True,Com Promocao,CLI1799,True,6,False,False,False,True
2,3,2025-03-08,2025,3,Marco,2025-03,T1,Sabado,8,15:15,15,Tarde,Pote,Baunilha,4,21.76,21.76,5.44,5.44,Câmara,PA,Parceiro,False,Sem Promocao,CLI9914,True,9,False,False,False,True
3,4,2025-08-13,2025,8,Agosto,2025-08,T3,Quarta,13,21:45,21,Noite,Casquinha,Cookies,3,30.87,30.87,10.29,10.29,Carvalho De Pereira,PR,Loja Física,True,Com Promocao,CLI2071,True,8,False,False,False,True
4,5,2025-04-07,2025,4,Abril,2025-04,T2,Segunda,7,11:45,11,Manha,Picolé,Cookies,4,26.59,26.59,6.65,6.65,Moraes,MA,Loja Física,True,Com Promocao,CLI9549,True,6,False,False,False,True


## Estatisticas Da Base Processada

In [10]:
processed[["quantidade_vendida", "receita_transacao", "valor_unitario_medio"]].describe().T

,count,mean,std,min,25%,50%,75%,max
quantidade_vendida,"48,491.00",3.08,1.43,1.00,2.00,3.00,4.00,6.00
receita_transacao,"48,491.00",28.17,16.29,3.60,14.51,25.64,38.76,89.88
valor_unitario_medio,"48,491.00",9.14,2.83,3.52,6.78,9.01,11.31,15.00


## Decisoes Principais

- `sabor` e `cidade` nulos foram preenchidos como `Nao Informado` e receberam flags de origem.
- Registros com `valor_total` nulo, `valor_total <= 0` ou `quantidade <= 0` foram excluidos da base processada e preservados em auditoria.
- A camada processada aplica nomes aprovados para Power BI, como `receita_transacao`, `quantidade_vendida`, `valor_unitario_medio`, `status_promocao` e `flag_registro_valido_powerbi`.
- Datas e horas foram convertidas e validadas.
- Categorias de produto, sabor, canal e estado foram validadas contra os dominios esperados.
- Outliers de `valor_total` foram sinalizados, mas mantidos, pois podem representar vendas reais maiores.
- A base final nao possui nulos, duplicidade de `id_transacao`, valores monetarios nao positivos ou quantidades nao positivas.